# Task 3 (v2, FIXED): GNN–BERT Fusion for Multi-Context Music Understanding

## What was broken in v1
Your ablation table showed `gnn_only` completely dead (Macro-F1=0.000, ROC-AUC=0.492 ≈ random),
which dragged down `concat` and `cross_attention` too. Root cause: **`build_segment_graph`
normalized each graph using its own mean/std** (computed across only its ~5-6 segments) — this
erases the between-clip differences (loudness, timbre, spectral energy) that actually
distinguish genres/instruments, leaving the GNN with only noise. Combined with 3 message-passing
layers over a 5-6 node graph (oversmoothing), embeddings collapsed to near-identical vectors for
every clip.

Separately, your per-tag table showed tags like `harp` with **ROC-AUC=0.887 but F1=0.000** —
the model genuinely ranks harp clips higher, but the fixed 0.5 threshold never triggers for a
rare tag. AUC-PR/ROC-AUC are threshold-free and showed real signal; Macro-F1 was threshold-broken.

## What v2 fixes
1. **Global feature normalization**: node features are cached **raw**, then normalized using
   mean/std computed once across the *training set* — preserves inter-clip signal.
2. **Shallower GNN** (2 layers instead of 3) to reduce oversmoothing on tiny 5-6 node graphs.
3. **Per-tag class-weighted loss** (`pos_weight`) so rare tags aren't suppressed to near-zero probability by default.
4. **Per-tag threshold tuning on the validation set** — the single biggest lever for Macro-F1/Micro-F1, since it doesn't change the model at all, just how probabilities are converted to predictions.

> Note: since the graph-feature caching format changed (raw vs. pre-normalized), this uses a
> **new cache directory** — your old cached `.pt` files are incompatible and won't be reused.


## 0. Environment setup

`torch-geometric` needs to match your installed torch/CUDA build. Run this once; if it fails,
check `torch.__version__` first and adjust the index URL accordingly.


In [ ]:
# Run once. Uncomment if torch_geometric isn't installed yet.
# import torch
# print(torch.__version__, torch.version.cuda)
# !pip install torch_geometric
# (PyG's core conv layers work without torch-scatter/torch-sparse for graphs this small —
#  our graphs have ~5-6 nodes each, so the pure-PyTorch fallback path is plenty fast.)


In [ ]:
import os, re, json, random, math, ast, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import librosa

from sklearn.metrics import f1_score, average_precision_score, roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

from torch_geometric.data import Data, Batch
from torch_geometric.nn import SAGEConv, global_mean_pool

warnings.filterwarnings("ignore")
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 1. Config


In [ ]:
class CFG:
    ANNOTATIONS_CSV = r"E:\CSE715\Datasets\MagnaTagATune\annotations_final.csv"
    AUDIO_ROOT = r"E:\CSE715\Datasets\MagnaTagATune\MagnaTagATune"
    GRAPH_CACHE_DIR = r"E:\CSE715\Datasets\MagnaTagATune\graph_cache_v2"   # NEW dir — raw (unnormalized) features
    OUTPUT_DIR = "./results_task3_v2"

    TOP_K_TAGS = 50
    SAMPLE_SIZE = 6000   # set to None for the full ~20k tagged clips once this runs cleanly

    SR = 16000
    CLIP_DURATION = 29.0
    SEGMENT_DURATION = 5.0
    SIMILARITY_THRESHOLD = 0.85

    MODEL_NAME = "distilbert-base-uncased"
    MAX_TEXT_LEN = 48
    GNN_HIDDEN = 128
    GNN_LAYERS = 2          # was 3 — reduced to curb oversmoothing on 5-6 node graphs
    FUSION_HIDDEN = 256
    DROPOUT = 0.3

    BATCH_SIZE = 32
    MAX_EPOCHS = 15
    PATIENCE = 3
    LR_BACKBONE = 2e-5
    LR_OTHER = 1e-3
    WEIGHT_DECAY = 0.03
    WARMUP_RATIO = 0.06
    NUM_UNFROZEN_BERT_LAYERS = 2

    POS_WEIGHT_CAP = 15.0   # cap on per-tag positive class weight, avoids extreme-imbalance blowups
    THRESH_GRID = np.arange(0.05, 0.95, 0.05)  # candidate thresholds for per-tag tuning

    SEED = 42
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    USE_AMP = torch.cuda.is_available()

os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(CFG.OUTPUT_DIR, "plots"), exist_ok=True)
os.makedirs(CFG.GRAPH_CACHE_DIR, exist_ok=True)
random.seed(CFG.SEED); np.random.seed(CFG.SEED); torch.manual_seed(CFG.SEED)


## 2. Load annotations, merge known synonym tags, select top-50

Merging synonym columns (`classical`+`clasical`, `male`+`male vocal`+`male voice`, etc.)
before ranking by frequency avoids fragmenting one concept across several sparse columns —
standard practice for this exact dataset.


In [ ]:
df = pd.read_csv(CFG.ANNOTATIONS_CSV, sep='\t')
tag_cols_raw = [c for c in df.columns if c not in ('clip_id', 'mp3_path')]
print(f"Loaded {len(df)} clips, {len(tag_cols_raw)} raw tag columns")

SYNONYM_GROUPS = {
    'classical':      ['classical', 'clasical'],
    'no vocal':       ['no vocal', 'no vocals', 'no voice', 'no singer', 'no singing'],
    'male':           ['man', 'male', 'male vocal', 'male voice'],
    'female':         ['woman', 'female', 'female vocal', 'female voice'],
    'synth':          ['synth', 'synthesizer'],
    'harpsichord':    ['harpsichord', 'harpsicord'],
    'electric guitar':['electric guitar', 'electric'],
    'sitar':          ['sitar', 'sittar'],
}

merged = pd.DataFrame(index=df.index)
used = set()
for canonical, group in SYNONYM_GROUPS.items():
    present = [g for g in group if g in tag_cols_raw]
    if present:
        merged[canonical] = (df[present].sum(axis=1) > 0).astype(int)
        used.update(present)

for c in tag_cols_raw:
    if c not in used:
        merged[c] = df[c]

print(f"After synonym merge: {merged.shape[1]} tag columns (from {len(tag_cols_raw)})")

tag_freq = merged.sum(axis=0).sort_values(ascending=False)
kept_tags = tag_freq.head(CFG.TOP_K_TAGS).index.tolist()
print(f"\nSelected top-{CFG.TOP_K_TAGS} tags. Frequency range: "
      f"{tag_freq[kept_tags].min()} - {tag_freq[kept_tags].max()}")
print(tag_freq.head(15))

df_tags = merged[kept_tags].copy()
df_tags['clip_id'] = df['clip_id']
df_tags['mp3_path'] = df['mp3_path']

# Drop clips with no positive tag among the kept top-50
has_tag = df_tags[kept_tags].sum(axis=1) > 0
df_tags = df_tags[has_tag].reset_index(drop=True)
print(f"\nClips remaining after dropping zero-tag rows: {len(df_tags)}")


## 3. Match clips to local audio files


In [ ]:
def resolve_audio_path(mp3_path, root):
    return os.path.join(root, mp3_path.replace('/', os.sep))

df_tags['audio_path'] = df_tags['mp3_path'].apply(lambda p: resolve_audio_path(p, CFG.AUDIO_ROOT))
exists_mask = df_tags['audio_path'].progress_apply(os.path.isfile) if hasattr(df_tags['audio_path'], 'progress_apply') else df_tags['audio_path'].apply(os.path.isfile)
print(f"Matched {exists_mask.sum()} / {len(df_tags)} audio files on disk "
      f"({100*exists_mask.mean():.1f}%).")

df_tags = df_tags[exists_mask].reset_index(drop=True)
if df_tags['mp3_path'].str.len().gt(0).all():
    df_tags['folder'] = df_tags['mp3_path'].str.split('/').str[0]
print(df_tags['folder'].value_counts().sort_index())


## 4. Parse pseudo-text from filenames (the BERT input)

MagnaTagATune filenames encode `artist-album-tracknum-title-startframe-endframe.mp3`. We
strip the frame-range suffix and turn underscores/hyphens into readable words.


In [ ]:
def parse_pseudo_text(mp3_path):
    fname = mp3_path.split('/')[-1].replace('.mp3', '')
    # strip trailing numeric frame-range tokens, e.g. "...-30-59"
    parts = fname.split('-')
    while len(parts) > 1 and parts[-1].isdigit():
        parts = parts[:-1]
    text = ' '.join(parts)
    text = text.replace('_', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text if text else "unknown track"

df_tags['text'] = df_tags['mp3_path'].apply(parse_pseudo_text)
df_tags[['mp3_path', 'text']].head(5)


## 5. Optional subsampling + official-style split by hex folder

We split by folder prefix (a common convention for this dataset): folders `0-9,a,b` → train,
`c,d` → validation, `e,f` → test. This avoids any single artist/album's segments (which share
a folder only coincidentally via hashing, not by album) dominating one split disproportionately
relative to standard MTT benchmarks.


In [ ]:
if CFG.SAMPLE_SIZE is not None and len(df_tags) > CFG.SAMPLE_SIZE:
    df_tags = df_tags.sample(n=CFG.SAMPLE_SIZE, random_state=CFG.SEED).reset_index(drop=True)
    print(f"Subsampled to {len(df_tags)} clips for faster iteration (CFG.SAMPLE_SIZE={CFG.SAMPLE_SIZE}).")

train_folders = set(list('0123456789ab'))
val_folders = set(list('cd'))
test_folders = set(list('ef'))

train_df = df_tags[df_tags['folder'].isin(train_folders)].reset_index(drop=True)
val_df = df_tags[df_tags['folder'].isin(val_folders)].reset_index(drop=True)
test_df = df_tags[df_tags['folder'].isin(test_folders)].reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")


## 6. Audio → segment graph construction

Each 29s clip is split into ~5s segments. Node features = mean+std pooled chroma (12-d) and
MFCC (13-d) per segment (50-d total). Edges: temporal adjacency (consecutive segments always
connected) + similarity edges wherever cosine similarity of node features exceeds
`SIMILARITY_THRESHOLD` (captures repeated/similar sections, e.g. verse-verse).
Graphs are cached to disk as `.pt` files — this is the slow one-time step; reruns skip
clips already cached.


In [ ]:
def build_segment_graph(audio_path, sr=CFG.SR, seg_dur=CFG.SEGMENT_DURATION,
                         sim_thresh=CFG.SIMILARITY_THRESHOLD):
    """Returns RAW (unnormalized) node features. Normalization now happens globally,
    using train-set statistics, applied later in FusionDataset — see the next section."""
    y, _ = librosa.load(audio_path, sr=sr, mono=True)
    total_dur = len(y) / sr
    n_segments = max(2, int(total_dur // seg_dur))
    seg_len = len(y) // n_segments

    node_feats = []
    for i in range(n_segments):
        start, end = i * seg_len, (i + 1) * seg_len if i < n_segments - 1 else len(y)
        seg = y[start:end]
        if len(seg) < sr * 0.5:
            continue
        chroma = librosa.feature.chroma_stft(y=seg, sr=sr, n_fft=1024, hop_length=512)
        mfcc = librosa.feature.mfcc(y=seg, sr=sr, n_mfcc=13, n_fft=1024, hop_length=512)
        feat = np.concatenate([
            chroma.mean(axis=1), chroma.std(axis=1),
            mfcc.mean(axis=1), mfcc.std(axis=1),
        ])
        node_feats.append(feat)

    if len(node_feats) < 2:
        return None

    x = np.stack(node_feats).astype(np.float32)   # RAW — no normalization here anymore

    n = x.shape[0]
    edges = set()
    for i in range(n - 1):
        edges.add((i, i + 1)); edges.add((i + 1, i))

    # similarity edges computed on raw features (cosine similarity is scale-invariant per-vector anyway)
    norm_x = x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-8)
    sim = norm_x @ norm_x.T
    for i in range(n):
        for j in range(n):
            if i != j and sim[i, j] > sim_thresh:
                edges.add((i, j))

    edge_index = np.array(list(edges)).T if edges else np.zeros((2, 0), dtype=np.int64)
    return Data(x=torch.tensor(x, dtype=torch.float32),
                edge_index=torch.tensor(edge_index, dtype=torch.long))


def get_or_build_graph(clip_id, audio_path):
    cache_path = os.path.join(CFG.GRAPH_CACHE_DIR, f"{clip_id}.pt")
    if os.path.exists(cache_path):
        return torch.load(cache_path, weights_only=False)
    graph = build_segment_graph(audio_path)
    if graph is not None:
        torch.save(graph, cache_path)
    return graph


In [ ]:
# One-time (resumable) caching pass over every clip we plan to use.
all_used_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
failed_clips = []

for _, row in tqdm(all_used_df.iterrows(), total=len(all_used_df), desc="Building/caching audio graphs"):
    cache_path = os.path.join(CFG.GRAPH_CACHE_DIR, f"{row['clip_id']}.pt")
    if os.path.exists(cache_path):
        continue
    try:
        g = build_segment_graph(row['audio_path'])
        if g is None:
            failed_clips.append(row['clip_id'])
        else:
            torch.save(g, cache_path)
    except Exception as e:
        failed_clips.append(row['clip_id'])

print(f"Failed/unusable clips: {len(failed_clips)} / {len(all_used_df)}")

# Drop any clip whose graph failed to build
def filter_valid(d):
    return d[~d['clip_id'].isin(failed_clips)].reset_index(drop=True)

train_df = filter_valid(train_df)
val_df = filter_valid(val_df)
test_df = filter_valid(test_df)
print(f"Final — Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")


## 6b. Compute GLOBAL node-feature normalization stats (the actual fix)

Computed once from the **training set only** (never touch val/test when computing
normalization stats — that would be its own form of leakage), then applied to every split.


In [ ]:
train_feats = []
for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Collecting train features for global stats"):
    g = get_or_build_graph(row['clip_id'], row['audio_path'])
    if g is not None:
        train_feats.append(g.x.numpy())

all_train_feats = np.concatenate(train_feats, axis=0)
NODE_MEAN = torch.tensor(all_train_feats.mean(axis=0), dtype=torch.float32)
NODE_STD = torch.tensor(all_train_feats.std(axis=0) + 1e-6, dtype=torch.float32)
print("Global node-feature mean/std computed from", all_train_feats.shape[0], "training segments.")
print("Mean range:", NODE_MEAN.min().item(), "to", NODE_MEAN.max().item())
print("Std range:", NODE_STD.min().item(), "to", NODE_STD.max().item())

def normalize_graph(data, mean=NODE_MEAN, std=NODE_STD):
    data = data.clone()
    data.x = (data.x - mean) / std
    return data


## 7. Dataset + collate function (graph batching via PyTorch Geometric)


In [ ]:
class FusionDataset(Dataset):
    def __init__(self, df, tag_cols):
        self.df = df.reset_index(drop=True)
        self.tag_cols = tag_cols

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        graph = get_or_build_graph(row['clip_id'], row['audio_path'])
        graph = normalize_graph(graph)   # GLOBAL normalization applied here — the key fix
        enc = tokenizer(row['text'], truncation=True, max_length=CFG.MAX_TEXT_LEN,
                         padding='max_length', return_tensors='pt')
        label = torch.tensor(row[self.tag_cols].values.astype(np.float32))
        return {
            'graph': graph,
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label': label,
        }

def fusion_collate(batch):
    graphs = Batch.from_data_list([b['graph'] for b in batch])
    input_ids = torch.stack([b['input_ids'] for b in batch])
    attention_mask = torch.stack([b['attention_mask'] for b in batch])
    labels = torch.stack([b['label'] for b in batch])
    return {'graph': graphs, 'input_ids': input_ids, 'attention_mask': attention_mask, 'label': labels}

train_ds = FusionDataset(train_df, kept_tags)
val_ds = FusionDataset(val_df, kept_tags)
test_ds = FusionDataset(test_df, kept_tags)

train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True, collate_fn=fusion_collate, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=CFG.BATCH_SIZE, shuffle=False, collate_fn=fusion_collate, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=CFG.BATCH_SIZE, shuffle=False, collate_fn=fusion_collate, num_workers=0)
print(f"Batches — train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}")


## 8. Models — GNN encoder, BERT encoder (partially frozen), and 4 fusion modes

`fusion_mode` selects which architecture the shared training loop trains, satisfying the
assignment's required ablation: **bert_only, gnn_only, concat, cross_attention**.


In [ ]:
class GNNEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_layers):
        super().__init__()
        self.convs = nn.ModuleList()
        self.convs.append(SAGEConv(in_dim, hidden_dim))
        for _ in range(num_layers - 1):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim))
        self.out_dim = hidden_dim

    def forward(self, x, edge_index, batch):
        for conv in self.convs:
            x = F.relu(conv(x, edge_index))
        g = global_mean_pool(x, batch)   # graph-level readout g
        return g


class BertEncoder(nn.Module):
    def __init__(self, model_name, num_unfrozen_layers):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.out_dim = self.bert.config.hidden_size
        for p in self.bert.parameters():
            p.requires_grad = False
        total_layers = len(self.bert.transformer.layer)
        for layer in self.bert.transformer.layer[total_layers - num_unfrozen_layers:]:
            for p in layer.parameters():
                p.requires_grad = True

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        return out.last_hidden_state   # (B, L, H) — needed for cross-attention; CLS = [:,0,:]


class CrossAttentionFusion(nn.Module):
    """z = concat(g, softmax(QK^T/sqrt(d)) @ V), Q from graph embedding, K/V from BERT tokens."""
    def __init__(self, g_dim, text_dim, d_model=128):
        super().__init__()
        self.q_proj = nn.Linear(g_dim, d_model)
        self.k_proj = nn.Linear(text_dim, d_model)
        self.v_proj = nn.Linear(text_dim, d_model)
        self.scale = d_model ** 0.5
        self.out_dim = g_dim + d_model

    def forward(self, g, Htext, attention_mask):
        Q = self.q_proj(g).unsqueeze(1)            # (B, 1, d)
        K = self.k_proj(Htext)                      # (B, L, d)
        V = self.v_proj(Htext)                       # (B, L, d)
        scores = (Q @ K.transpose(1, 2)) / self.scale  # (B, 1, L)
        mask = (1.0 - attention_mask.unsqueeze(1).float()) * -1e9
        scores = scores + mask
        attn = torch.softmax(scores, dim=-1)
        context = (attn @ V).squeeze(1)              # (B, d)
        return torch.cat([g, context], dim=1)


class FusionModel(nn.Module):
    def __init__(self, num_labels, fusion_mode="cross_attention",
                 node_feat_dim=50, gnn_hidden=CFG.GNN_HIDDEN, gnn_layers=CFG.GNN_LAYERS,
                 model_name=CFG.MODEL_NAME, num_unfrozen=CFG.NUM_UNFROZEN_BERT_LAYERS,
                 fusion_hidden=CFG.FUSION_HIDDEN, dropout=CFG.DROPOUT):
        super().__init__()
        assert fusion_mode in ("bert_only", "gnn_only", "concat", "cross_attention")
        self.fusion_mode = fusion_mode

        self.gnn = GNNEncoder(node_feat_dim, gnn_hidden, gnn_layers) if fusion_mode != "bert_only" else None
        self.bert = BertEncoder(model_name, num_unfrozen) if fusion_mode != "gnn_only" else None

        if fusion_mode == "bert_only":
            in_dim = self.bert.out_dim
        elif fusion_mode == "gnn_only":
            in_dim = self.gnn.out_dim
        elif fusion_mode == "concat":
            in_dim = self.gnn.out_dim + self.bert.out_dim
        elif fusion_mode == "cross_attention":
            self.cross_attn = CrossAttentionFusion(self.gnn.out_dim, self.bert.out_dim, d_model=fusion_hidden // 2)
            in_dim = self.cross_attn.out_dim

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(in_dim, fusion_hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(fusion_hidden, num_labels)
        )

    def forward(self, graph_batch, input_ids, attention_mask):
        g = None
        Htext = None
        if self.gnn is not None:
            g = self.gnn(graph_batch.x, graph_batch.edge_index, graph_batch.batch)
        if self.bert is not None:
            Htext = self.bert(input_ids, attention_mask)

        if self.fusion_mode == "bert_only":
            z = Htext[:, 0, :]
        elif self.fusion_mode == "gnn_only":
            z = g
        elif self.fusion_mode == "concat":
            z = torch.cat([g, Htext[:, 0, :]], dim=1)
        elif self.fusion_mode == "cross_attention":
            z = self.cross_attn(g, Htext, attention_mask)

        z = self.dropout(z)
        logits = self.classifier(z)
        return logits, z   # return z too, for t-SNE later

NODE_FEAT_DIM = train_ds[0]['graph'].x.shape[1]
print("Node feature dim:", NODE_FEAT_DIM)


## 9. Shared training loop (with early stopping) — reused across all 4 ablations


In [ ]:
def compute_pos_weight(df, tag_cols, cap=CFG.POS_WEIGHT_CAP):
    pos = df[tag_cols].sum(axis=0).values.astype(np.float32)
    neg = len(df) - pos
    weight = np.clip(neg / np.maximum(pos, 1), 1.0, cap)
    return torch.tensor(weight, dtype=torch.float32)

POS_WEIGHT = compute_pos_weight(train_df, kept_tags).to(CFG.DEVICE)
print("Pos-weight range:", POS_WEIGHT.min().item(), "-", POS_WEIGHT.max().item())


def tune_thresholds(probs, labels, grid=CFG.THRESH_GRID):
    """Per-tag threshold that maximizes F1 on the given (validation) set."""
    n_tags = labels.shape[1]
    best_thresh = np.full(n_tags, 0.5)
    for k in range(n_tags):
        if labels[:, k].sum() == 0:
            continue
        best_f1, best_t = -1, 0.5
        for t in grid:
            f1 = f1_score(labels[:, k], (probs[:, k] >= t).astype(int), zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        best_thresh[k] = best_t
    return best_thresh


def evaluate_model(model, loader, criterion, thresholds=None):
    model.eval()
    total_loss = 0.0
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            graph = batch['graph'].to(CFG.DEVICE)
            input_ids = batch['input_ids'].to(CFG.DEVICE)
            attn = batch['attention_mask'].to(CFG.DEVICE)
            labels = batch['label'].to(CFG.DEVICE)

            with torch.cuda.amp.autocast(enabled=CFG.USE_AMP):
                logits, _ = model(graph, input_ids, attn)
                loss = criterion(logits, labels)

            total_loss += loss.item() * labels.size(0)
            all_probs.append(torch.sigmoid(logits).float().cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    probs = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    th = thresholds if thresholds is not None else np.full(probs.shape[1], 0.5)
    preds = (probs >= th[None, :]).astype(int)

    macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)
    micro_f1 = f1_score(labels, preds, average='micro', zero_division=0)
    try:
        auc_pr = average_precision_score(labels, probs, average='macro')
    except ValueError:
        auc_pr = float('nan')
    try:
        roc_auc = roc_auc_score(labels, probs, average='macro')
    except ValueError:
        roc_auc = float('nan')

    return total_loss / len(loader.dataset), macro_f1, micro_f1, auc_pr, roc_auc, probs, labels


In [ ]:
def train_fusion_model(fusion_mode, max_epochs=CFG.MAX_EPOCHS, patience=CFG.PATIENCE):
    print(f"\n{'='*70}\nTraining fusion_mode = {fusion_mode}\n{'='*70}")
    model = FusionModel(num_labels=len(kept_tags), fusion_mode=fusion_mode,
                         node_feat_dim=NODE_FEAT_DIM).to(CFG.DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT)   # class-weighted — key fix #2
    backbone_params = [p for n, p in model.named_parameters() if p.requires_grad and ('bert' in n)]
    other_params = [p for n, p in model.named_parameters() if p.requires_grad and ('bert' not in n)]
    param_groups = []
    if backbone_params:
        param_groups.append({'params': backbone_params, 'lr': CFG.LR_BACKBONE})
    param_groups.append({'params': other_params, 'lr': CFG.LR_OTHER})
    optimizer = torch.optim.AdamW(param_groups, weight_decay=CFG.WEIGHT_DECAY)

    total_steps = len(train_loader) * max_epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(CFG.WARMUP_RATIO*total_steps), total_steps)
    scaler = torch.cuda.amp.GradScaler(enabled=CFG.USE_AMP)

    history = {'train_loss': [], 'val_loss': [], 'val_macro_f1': [], 'val_auc_pr': [], 'val_roc_auc': []}
    best_macro_f1, best_state, no_improve = -1, None, 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"[{fusion_mode}] Epoch {epoch}/{max_epochs}")
        for batch in pbar:
            graph = batch['graph'].to(CFG.DEVICE)
            input_ids = batch['input_ids'].to(CFG.DEVICE)
            attn = batch['attention_mask'].to(CFG.DEVICE)
            labels = batch['label'].to(CFG.DEVICE)

            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=CFG.USE_AMP):
                logits, _ = model(graph, input_ids, attn)
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            running_loss += loss.item() * labels.size(0)
            pbar.set_postfix(loss=loss.item())

        train_loss = running_loss / len(train_loader.dataset)
        # Track raw (0.5-threshold) val Macro-F1 during training just to pick the best checkpoint —
        # final reported numbers use tuned thresholds, computed once after training below.
        val_loss, val_mf1, val_mif1, val_aucpr, val_rocauc, _, _ = evaluate_model(model, val_loader, criterion)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_macro_f1'].append(val_mf1)
        history['val_auc_pr'].append(val_aucpr)
        history['val_roc_auc'].append(val_rocauc)

        print(f"Epoch {epoch}: train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
              f"val_macroF1(@0.5)={val_mf1:.4f} val_AUC-PR={val_aucpr:.4f} val_ROC-AUC={val_rocauc:.4f}")

        # Model selection uses AUC-PR, not the 0.5-threshold F1 — AUC-PR isn't distorted by
        # miscalibrated thresholds the way Macro-F1 is, so it's a more reliable stopping signal.
        if val_aucpr > best_macro_f1:
            best_macro_f1 = val_aucpr
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch}.")
                break

    model.load_state_dict(best_state)

    # ---- Threshold tuning: find per-tag best threshold on VALIDATION set, apply to TEST ----
    _, _, _, _, _, val_probs, val_labels = evaluate_model(model, val_loader, criterion)
    tuned_thresholds = tune_thresholds(val_probs, val_labels)

    test_loss, test_mf1_default, test_mif1_default, test_aucpr, test_rocauc, test_probs, test_labels = \
        evaluate_model(model, test_loader, criterion, thresholds=None)
    test_preds_tuned = (test_probs >= tuned_thresholds[None, :]).astype(int)
    test_mf1 = f1_score(test_labels, test_preds_tuned, average='macro', zero_division=0)
    test_mif1 = f1_score(test_labels, test_preds_tuned, average='micro', zero_division=0)

    print(f"\n[{fusion_mode}] TEST — Macro-F1 @0.5: {test_mf1_default:.4f} | Macro-F1 (tuned thresholds): {test_mf1:.4f}")
    print(f"[{fusion_mode}] TEST — Micro-F1 @0.5: {test_mif1_default:.4f} | Micro-F1 (tuned thresholds): {test_mif1:.4f}")
    print(f"[{fusion_mode}] TEST — AUC-PR: {test_aucpr:.4f} | ROC-AUC: {test_rocauc:.4f}")

    return {
        'model': model, 'history': history, 'tuned_thresholds': tuned_thresholds,
        'test_macro_f1': test_mf1, 'test_micro_f1': test_mif1,
        'test_macro_f1_default': test_mf1_default, 'test_micro_f1_default': test_mif1_default,
        'test_auc_pr': test_aucpr, 'test_roc_auc': test_rocauc,
        'test_probs': test_probs, 'test_labels': test_labels,
    }


## 10. Run the required ablation: BERT-only, GNN-only, early-concat, cross-attention fusion


In [ ]:
results = {}
for mode in ["bert_only", "gnn_only", "concat", "cross_attention"]:
    results[mode] = train_fusion_model(mode)


## 11. Ablation comparison table (deliverable)


In [ ]:
ablation_table = pd.DataFrame([
    {'Model': mode,
     'Macro-F1 (tuned th)': r['test_macro_f1'], 'Macro-F1 (@0.5)': r['test_macro_f1_default'],
     'Micro-F1 (tuned th)': r['test_micro_f1'], 'Micro-F1 (@0.5)': r['test_micro_f1_default'],
     'AUC-PR': r['test_auc_pr'], 'ROC-AUC': r['test_roc_auc']}
    for mode, r in results.items()
])
ablation_table = ablation_table.sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
ablation_table


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ablation_table.set_index('Model')[['Macro-F1 (tuned th)', 'AUC-PR', 'ROC-AUC']].plot(kind='bar', ax=ax)
ax.set_ylabel('Score'); ax.set_title('Ablation: BERT-only vs GNN-only vs Concat vs Cross-Attention')
ax.set_ylim(0, 1)
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(os.path.join(CFG.OUTPUT_DIR, 'plots', 'ablation_comparison.png'), dpi=150)
plt.show()


## 12. Best model — training curves and per-tag breakdown


In [ ]:
best_mode = ablation_table.iloc[0]['Model']
best_result = results[best_mode]
print(f"Best model by ROC-AUC: {best_mode}")

hist = best_result['history']
epochs_ran = range(1, len(hist['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(epochs_ran, hist['train_loss'], marker='o', label='train loss')
axes[0].plot(epochs_ran, hist['val_loss'], marker='o', label='val loss')
axes[0].set_xlabel('Epoch'); axes[0].set_title(f'Loss — {best_mode}'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_ran, hist['val_macro_f1'], marker='o', label='Val Macro-F1')
axes[1].plot(epochs_ran, hist['val_auc_pr'], marker='o', label='Val AUC-PR')
axes[1].plot(epochs_ran, hist['val_roc_auc'], marker='o', label='Val ROC-AUC')
axes[1].set_xlabel('Epoch'); axes[1].set_title('Validation metrics'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CFG.OUTPUT_DIR, 'plots', 'best_model_curves.png'), dpi=150)
plt.show()


In [ ]:
from sklearn.metrics import precision_recall_fscore_support

probs, labels = best_result['test_probs'], best_result['test_labels']
tuned_th = best_result['tuned_thresholds']
preds = (probs >= tuned_th[None, :]).astype(int)   # tuned per-tag thresholds, not a blanket 0.5

precision, recall, f1s, support = precision_recall_fscore_support(labels, preds, average=None, zero_division=0)
per_tag_aucpr = [average_precision_score(labels[:, k], probs[:, k]) if labels[:, k].sum() > 0 else np.nan
                 for k in range(len(kept_tags))]
per_tag_rocauc = [roc_auc_score(labels[:, k], probs[:, k]) if len(set(labels[:, k])) > 1 else np.nan
                   for k in range(len(kept_tags))]

per_tag_df = pd.DataFrame({
    'tag': kept_tags, 'precision': precision, 'recall': recall, 'f1': f1s,
    'support': support, 'AUC-PR': per_tag_aucpr, 'ROC-AUC': per_tag_rocauc,
    'tuned_threshold': tuned_th,
}).sort_values('ROC-AUC', ascending=False)
per_tag_df


## 13. t-SNE of fused embeddings (colored by top-5 most frequent tags present)


In [ ]:
best_model = best_result['model']
best_model.eval()

all_z, all_labels_for_tsne = [], []
with torch.no_grad():
    for batch in test_loader:
        graph = batch['graph'].to(CFG.DEVICE)
        input_ids = batch['input_ids'].to(CFG.DEVICE)
        attn = batch['attention_mask'].to(CFG.DEVICE)
        _, z = best_model(graph, input_ids, attn)
        all_z.append(z.cpu().numpy())
        all_labels_for_tsne.append(batch['label'].numpy())

Z = np.concatenate(all_z)
L = np.concatenate(all_labels_for_tsne)

top5_tags_idx = [kept_tags.index(t) for t in tag_freq.head(5).index if t in kept_tags][:5]
color_labels = np.argmax(L[:, top5_tags_idx], axis=1)  # dominant of the top-5 tags per clip (for coloring only)

tsne = TSNE(n_components=2, random_state=CFG.SEED, perplexity=30, init='pca')
Z_2d = tsne.fit_transform(Z)

fig, ax = plt.subplots(figsize=(8, 7))
top5_names = [kept_tags[i] for i in top5_tags_idx]
scatter = ax.scatter(Z_2d[:, 0], Z_2d[:, 1], c=color_labels, cmap='tab10', alpha=0.6, s=15)
handles, _ = scatter.legend_elements()
ax.legend(handles, top5_names, title="Dominant tag (of top-5)")
ax.set_title(f't-SNE of fused embeddings z ({best_mode})')
plt.tight_layout()
plt.savefig(os.path.join(CFG.OUTPUT_DIR, 'plots', 'tsne_fusion_embeddings.png'), dpi=150)
plt.show()


## 14. Case studies — graph structure + pseudo-caption alignment (deliverable)


In [ ]:
sample_idx = np.random.RandomState(CFG.SEED).choice(len(test_df), size=3, replace=False)

for i in sample_idx:
    row = test_df.iloc[i]
    graph = get_or_build_graph(row['clip_id'], row['audio_path'])
    true_tags_i = [t for t in kept_tags if row[t] == 1]

    enc = tokenizer(row['text'], truncation=True, max_length=CFG.MAX_TEXT_LEN, padding='max_length', return_tensors='pt').to(CFG.DEVICE)
    g_batch = Batch.from_data_list([graph]).to(CFG.DEVICE)
    with torch.no_grad():
        logits, _ = best_model(g_batch, enc['input_ids'], enc['attention_mask'])
        probs_i = torch.sigmoid(logits).cpu().numpy()[0]
    top5_pred = [(kept_tags[j], round(float(probs_i[j]), 3)) for j in np.argsort(-probs_i)[:5]]

    print("-" * 90)
    print(f"Pseudo-text : {row['text']}")
    print(f"Graph       : {graph.x.shape[0]} nodes (segments), {graph.edge_index.shape[1]} directed edges")
    print(f"True tags   : {true_tags_i}")
    print(f"Top-5 pred  : {top5_pred}")


## 15. Save everything


In [ ]:
summary = {
    'ablation_table': ablation_table.to_dict(orient='records'),
    'best_model': best_mode,
    'num_tags': len(kept_tags),
    'tags': kept_tags,
    'train_size': len(train_df), 'val_size': len(val_df), 'test_size': len(test_df),
}
with open(os.path.join(CFG.OUTPUT_DIR, 'task3_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

per_tag_df.to_csv(os.path.join(CFG.OUTPUT_DIR, 'per_tag_metrics.csv'), index=False)
torch.save(best_model.state_dict(), os.path.join(CFG.OUTPUT_DIR, f'best_model_{best_mode}.pt'))
print("Saved task3_summary.json, per_tag_metrics.csv, best_model_*.pt")


## Notes on pushing metrics further

- **`CFG.SAMPLE_SIZE`**: set to `None` for the full ~20k tagged clips once the pipeline runs
  cleanly on the subset — more data is the single biggest lever for the rare tags in the
  per-tag table above.
- **`CFG.NUM_UNFROZEN_BERT_LAYERS`**: try 3-4 if val loss isn't diverging (watch the loss plot
  the same way we diagnosed Task 1's overfitting).
- **Segment count**: `SEGMENT_DURATION=5.0` gives ~5-6 nodes/graph from a 29s clip; shorter
  segments (3s) give richer graphs at the cost of noisier per-node features — worth an ablation
  if GNN-only underperforms BERT-only in your ablation table.
- If a handful of tags have `support` near zero in the per-tag table, that's a genuine
  long-tail limit, not a training bug — report it honestly in your per-tag analysis, same as
  we did for Task 1.
